In [1]:
# import os
# import random #데이터 샘플링
# from collections import Counter # count 용도

import numpy as np
import pandas as pd

from tqdm import tqdm

# 결측치 확인
import missingno as msno

import warnings
warnings.filterwarnings('ignore')

# 시각화
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
plt.style.use('fivethirtyeight')

# 한글, 마이너스 깨짐 방지
from matplotlib import rc, font_manager, rcParams
font=font_manager.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()
rc('font', family = font)
rcParams['axes.unicode_minus'] = False

# # 지도
# # from geopy import distance # 거리 계산
# # import geopy.distance
# import folium
# from folium.plugins import HeatMap

# # plotly
# import ipywidgets as widgets
# from ipywidgets import interact

# # 이걸 설정하면 Multiple Output이 가능함
# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "all"

# import chart_studio.plotly as py 
# import plotly.express as px
# import cufflinks as cf 
# cf.go_offline(connected=True)

# import plotly.graph_objects as go

## 데이터 불러오기

In [3]:
df = pd.read_csv('./dataset/restaurant_raw.csv')
df.head()

,name,category,address,score,eval_cnt,review_cnt
0,진미평양냉면,냉면,(지번) 논현동 115-10,3.7,204건,리뷰 532
1,대가방 본점,중화요리,(지번) 논현동 99-7,3.3,86건,리뷰 101
2,세종한우 2호점,"육류,고기",(지번) 논현동 216-10,3.0,27건,리뷰 42
3,크래버대게나라 강남점,"게,대게",(지번) 논현동 261,4.1,38건,리뷰 188
4,카페써드,커피전문점,(지번) 논현동 118-4,3.7,12건,리뷰 100


## 데이터 전처리

### 결측값 확인

In [4]:
df.isnull().sum()

name          0
category      0
address       0
score         4
eval_cnt      0
review_cnt    0
dtype: int64

In [5]:
df[df['score'].isnull()]

,name,category,address,score,eval_cnt,review_cnt
11,김수사,"초밥,롤",(지번) 논현동 4-19,NaN,(30),리뷰 128
26,김수사,"초밥,롤",(지번) 논현동 4-19,NaN,(30),리뷰 128
209,유림상회,술집,(지번) 논현동 165-17,NaN,(30),리뷰 82
294,천장지구,다방,(지번) 논현동 93-15,NaN,(30),리뷰 72


+ 확인하고 카테고리별 중앙값 또는 평균값으로 처리

### 중복 데이터 확인

In [7]:
df['name'].value_counts().head(20)

진미평양냉면              2
현우동                 2
대가방 본점              2
달맞이                 2
일일향 2호점             2
홍명                  2
김수사                 2
맛짱조개                2
대삼식당                2
리북집 논현직영점           2
한성칼국수               2
해몽                  2
카페써드                2
크래버대게나라 강남점         2
우정양곱창               2
세종한우 2호점            2
얼짱포차                1
용삼계탕                1
킹박스                 1
맛뜸최가뼈다귀해장국 논현1호점    1
Name: name, dtype: int64

In [9]:
# 2개인 것들 확인하기
df[df['name']=='진미평양냉면']

,name,category,address,score,eval_cnt,review_cnt
0,진미평양냉면,냉면,(지번) 논현동 115-10,3.7,204건,리뷰 532
15,진미평양냉면,냉면,(지번) 논현동 115-10,3.7,204건,리뷰 532


In [10]:
df[df['name']=='대삼식당']

,name,category,address,score,eval_cnt,review_cnt
59,대삼식당,삼겹살,(지번) 논현동 106-17,3.5,31건,리뷰 102
416,대삼식당,"육류,고기",(지번) 논현동 101-13,4.8,4건,리뷰 0


In [15]:
df[df['name']=='해몽']

,name,category,address,score,eval_cnt,review_cnt
5,해몽,"육류,고기",(지번) 논현동 107-11,4.2,45건,리뷰 173
20,해몽,"육류,고기",(지번) 논현동 107-11,4.2,45건,리뷰 173


In [16]:
# 중복값 제거
df.drop_duplicates('name', inplace=True)

In [17]:
df[df['name']=='대삼식당']

,name,category,address,score,eval_cnt,review_cnt
59,대삼식당,삼겹살,(지번) 논현동 106-17,3.5,31건,리뷰 102


In [18]:
df.shape

(499, 6)

In [20]:
# index 재정렬
df.index = range(len(df))
df

,name,category,address,score,eval_cnt,review_cnt
0,진미평양냉면,냉면,(지번) 논현동 115-10,3.7,204건,리뷰 532
1,대가방 본점,중화요리,(지번) 논현동 99-7,3.3,86건,리뷰 101
2,세종한우 2호점,"육류,고기",(지번) 논현동 216-10,3.0,27건,리뷰 42
3,크래버대게나라 강남점,"게,대게",(지번) 논현동 261,4.1,38건,리뷰 188
4,카페써드,커피전문점,(지번) 논현동 118-4,3.7,12건,리뷰 100
...,...,...,...,...,...,...
494,스윙부스,카페,(지번) 논현동 151-17,5.0,4건,리뷰 15
495,60계 서울논현점,치킨,(지번) 논현동 183-2,3.6,8건,리뷰 24
496,필스너하우스 청담점,"호프,요리주점",(지번) 논현동 96-12,4.6,7건,리뷰 15
497,웨이크업커피 2호점,카페,(지번) 논현동 86-4,5.0,1건,리뷰 3


### address 컬럼

+ '(지번) ' 삭제

In [22]:
# 반복문, 조건문 사용해서 지번 제거하기
for i in tqdm(df.index):
    if df.address[i][:5] == '(지번) ':
        df.address[i] = df.address[i][5:]

100%|██████████████████████████████████████████████████████████████████████████████| 499/499 [00:00<00:00, 2826.78it/s]


In [23]:
df

,name,category,address,score,eval_cnt,review_cnt
0,진미평양냉면,냉면,논현동 115-10,3.7,204건,리뷰 532
1,대가방 본점,중화요리,논현동 99-7,3.3,86건,리뷰 101
2,세종한우 2호점,"육류,고기",논현동 216-10,3.0,27건,리뷰 42
3,크래버대게나라 강남점,"게,대게",논현동 261,4.1,38건,리뷰 188
4,카페써드,커피전문점,논현동 118-4,3.7,12건,리뷰 100
...,...,...,...,...,...,...
494,스윙부스,카페,논현동 151-17,5.0,4건,리뷰 15
495,60계 서울논현점,치킨,논현동 183-2,3.6,8건,리뷰 24
496,필스너하우스 청담점,"호프,요리주점",논현동 96-12,4.6,7건,리뷰 15
497,웨이크업커피 2호점,카페,논현동 86-4,5.0,1건,리뷰 3


### eval_cnt 컬럼

+ '건' 제거

In [24]:
for i in tqdm(df.index):
    if df.eval_cnt[i][-1] != '건':
        print(i)

100%|█████████████████████████████████████████████████████████████████████████████| 499/499 [00:00<00:00, 45491.17it/s]

11
194
279


In [29]:
# 0점과 0건으로 처리
for i in [11, 194, 279]:
    df.loc[i, 'score'] = 0.0
    df.loc[i, 'eval_cnt'] = '0건'

In [30]:
df[df['eval_cnt']=='0건']

,name,category,address,score,eval_cnt,review_cnt
11,김수사,"초밥,롤",논현동 4-19,0.0,0건,리뷰 128
194,유림상회,술집,논현동 165-17,0.0,0건,리뷰 82
233,전광수커피하우스 학동사거리점,커피전문점,논현동 96-6,0.0,0건,리뷰 7
279,천장지구,다방,논현동 93-15,0.0,0건,리뷰 72
288,돼지연구소,"육류,고기",논현동 233,0.0,0건,리뷰 6
344,네버랜드,카페,논현동 180-2,0.0,0건,리뷰 7
362,커피렉라보 도산사거리점,카페,논현동 97-11,0.0,0건,리뷰 12
367,논현역2번출구,"육류,고기",논현동 144,0.0,0건,리뷰 2
432,대열빈,음식점,논현동 226,0.0,0건,리뷰 0
487,착한부자,도시락,논현동 193-27,0.0,0건,리뷰 46


In [31]:
# 해당 컬럼의 모든 레코드가 '건'으로 끝나는지 확인
df.eval_cnt.apply(lambda x: str(x)[-1]).value_counts()

건    499
Name: eval_cnt, dtype: int64

In [32]:
# 건 삭제
df['eval_cnt'] = df['eval_cnt'].apply(lambda x: str(x)[:-1])
df.head()

,name,category,address,score,eval_cnt,review_cnt
0,진미평양냉면,냉면,논현동 115-10,3.7,204,리뷰 532
1,대가방 본점,중화요리,논현동 99-7,3.3,86,리뷰 101
2,세종한우 2호점,"육류,고기",논현동 216-10,3.0,27,리뷰 42
3,크래버대게나라 강남점,"게,대게",논현동 261,4.1,38,리뷰 188
4,카페써드,커피전문점,논현동 118-4,3.7,12,리뷰 100


In [33]:
# 데이터 형식 변경
df['eval_cnt'] = df['eval_cnt'].astype(int)

### review_cnt 컬럼

+ '리뷰' 삭제
+ int로 데이터 타입 변경

In [36]:
# 리뷰 글자 확인
df['review_cnt'].apply(lambda x: str(x)[:3]).value_counts()

리뷰     499
Name: review_cnt, dtype: int64

In [37]:
# 리뷰 제거
df['review_cnt'] = df['review_cnt'].apply(lambda x: str(x)[3:])

In [38]:
df.head()

,name,category,address,score,eval_cnt,review_cnt
0,진미평양냉면,냉면,논현동 115-10,3.7,204,532
1,대가방 본점,중화요리,논현동 99-7,3.3,86,101
2,세종한우 2호점,"육류,고기",논현동 216-10,3.0,27,42
3,크래버대게나라 강남점,"게,대게",논현동 261,4.1,38,188
4,카페써드,커피전문점,논현동 118-4,3.7,12,100


In [40]:
df['review_cnt'].apply(lambda x: len(str(x))).value_counts()

2    344
1    102
3     52
5      1
Name: review_cnt, dtype: int64

In [41]:
df[df['review_cnt']=='1,009']

,name,category,address,score,eval_cnt,review_cnt
385,기대만족 강남본점,"족발,보쌈",논현동 181-14,2.0,12,"1,009"


In [44]:
df.loc[385, 'review_cnt'] = df.loc[385, 'review_cnt'].replace(',', '')
df.loc[385, 'review_cnt']

'1009'

In [45]:
# 데이터 타입 변경
df['review_cnt'] = df['review_cnt'].astype(int)

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        499 non-null    object 
 1   category    499 non-null    object 
 2   address     499 non-null    object 
 3   score       499 non-null    float64
 4   eval_cnt    499 non-null    int32  
 5   review_cnt  499 non-null    int32  
dtypes: float64(1), int32(2), object(3)
memory usage: 19.6+ KB


### 이상값 처리

In [47]:
df['eval_cnt'].value_counts().sort_index()

0      10
1      15
2      24
3      18
4      28
5      24
6      34
7      24
8      24
9      31
10     29
11     15
12     22
13     13
14     17
15     17
16     15
17      9
18      8
19      7
20      8
21      9
22      4
23      7
24      6
25      6
26      7
27      6
28      6
29      5
30      2
31      7
32      2
34      3
35      2
36      2
37      1
38      1
39      5
40      1
41      4
42      1
45      1
47      1
50      1
53      1
54      1
57      2
69      1
72      2
73      2
74      1
82      1
86      1
87      1
93      1
121     1
204     1
346     1
Name: eval_cnt, dtype: int64

In [48]:
df['review_cnt'].value_counts().sort_index()

0        2
1        4
2       10
3       10
4        8
        ..
263      1
358      1
405      1
532      1
1009     1
Name: review_cnt, Length: 129, dtype: int64

### 지리정보 추가하기

### 위도, 경도 추가

In [55]:
import googlemaps
gmap_key = '********'
gmaps = googlemaps.Client(key=gmap_key)

In [56]:
# error
error_num = 0

# 구글맵API를 활용하여 각 카페의 주소에 해당하는 지리 정보를 얻어오기
for row in tqdm(df.index):
    try:
        geo = gmaps.geocode(str(df.loc[row, 'address']))
        df.loc[row, 'lat'] = geo[0].get('geometry')['location']['lat']
        df.loc[row, 'lng'] = geo[0].get('geometry')['location']['lng']
    except:
        df.loc[row, 'lat'] = np.nan
        df.loc[row, 'lng'] = np.nan
        error_num += 1

print(error_num)
df.head()

100%|████████████████████████████████████████████████████████████████████████████████| 499/499 [01:49<00:00,  4.55it/s]

0


,name,category,address,score,eval_cnt,review_cnt,lat,lng
0,진미평양냉면,냉면,논현동 115-10,3.7,204,532,37.516107,127.036030
1,대가방 본점,중화요리,논현동 99-7,3.3,86,101,37.521115,127.038526
2,세종한우 2호점,"육류,고기",논현동 216-10,3.0,27,42,37.510889,127.032509
3,크래버대게나라 강남점,"게,대게",논현동 261,4.1,38,188,37.511590,127.036288
4,카페써드,커피전문점,논현동 118-4,3.7,12,100,37.517362,127.039830


In [57]:
df.head()

,name,category,address,score,eval_cnt,review_cnt,lat,lng
0,진미평양냉면,냉면,논현동 115-10,3.7,204,532,37.516107,127.036030
1,대가방 본점,중화요리,논현동 99-7,3.3,86,101,37.521115,127.038526
2,세종한우 2호점,"육류,고기",논현동 216-10,3.0,27,42,37.510889,127.032509
3,크래버대게나라 강남점,"게,대게",논현동 261,4.1,38,188,37.511590,127.036288
4,카페써드,커피전문점,논현동 118-4,3.7,12,100,37.517362,127.039830


In [ ]:
37.516107, 127.036030

### distance 컬럼 추가

In [58]:
from haversine import haversine

In [59]:
geo = gmaps.geocode(str('서울특별시 강남구 논현1동 논현로127길 13-11'))
print(geo[0].get('geometry')['location']['lat'])
print(geo[0].get('geometry')['location']['lng'])

37.5121248
127.030334


In [60]:
nexthub = (37.5121248, 127.030334)

In [61]:
for index in tqdm(df.index):
    df.loc[index, 'distance'] = haversine(nexthub, (df.loc[index, 'lat'], df.loc[index, 'lng']), unit='m')

df.head()

100%|██████████████████████████████████████████████████████████████████████████████| 499/499 [00:00<00:00, 2417.08it/s]


,name,category,address,score,eval_cnt,review_cnt,lat,lng,distance
0,진미평양냉면,냉면,논현동 115-10,3.7,204,532,37.516107,127.036030,669.698432
1,대가방 본점,중화요리,논현동 99-7,3.3,86,101,37.521115,127.038526,1233.413552
2,세종한우 2호점,"육류,고기",논현동 216-10,3.0,27,42,37.510889,127.032509,235.981638
3,크래버대게나라 강남점,"게,대게",논현동 261,4.1,38,188,37.511590,127.036288,528.555530
4,카페써드,커피전문점,논현동 118-4,3.7,12,100,37.517362,127.039830,1020.061748


## 저장하기

In [65]:
# df.to_csv('./dataset/restaurant_pre.csv', index=False)

In [66]:
pd.read_csv('./dataset/restaurant_pre.csv')

,name,category,address,score,eval_cnt,review_cnt,lat,lng,distance
0,진미평양냉면,냉면,논현동 115-10,3.7,204,532,37.516107,127.036030,669.698432
1,대가방 본점,중화요리,논현동 99-7,3.3,86,101,37.521115,127.038526,1233.413552
2,세종한우 2호점,"육류,고기",논현동 216-10,3.0,27,42,37.510889,127.032509,235.981638
3,크래버대게나라 강남점,"게,대게",논현동 261,4.1,38,188,37.511590,127.036288,528.555530
4,카페써드,커피전문점,논현동 118-4,3.7,12,100,37.517362,127.039830,1020.061748
...,...,...,...,...,...,...,...,...,...
494,스윙부스,카페,논현동 151-17,5.0,4,15,37.511115,127.030901,122.943185
495,60계 서울논현점,치킨,논현동 183-2,3.6,8,24,37.506444,127.024751,800.969261
496,필스너하우스 청담점,"호프,요리주점",논현동 96-12,4.6,7,15,37.521158,127.037377,1180.985491
497,웨이크업커피 2호점,카페,논현동 86-4,5.0,1,3,37.515219,127.031794,367.396766
